In [1]:
#geospatial clustering for developing separate rf models for each region
#clusters based on relationship of predictor variables with precipitation (target variable)
#we use stable climatology (30yrs) of predictor variables and precipitation(ERA5L)
#resample all variables to common grid (1km)
#fit rf models to derive weights of each predictor variable to precipitation
#perform weighted k means clustering 
#optimum cluster determined by elbow + shiloutte method


#UPDATE DATA RANGE 2000 TO 2020
#improve no data handling and change to era5l

In [2]:
import os
import glob
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import rioxarray as rxr
import xarray as xr
import matplotlib.pyplot as plt
from rasterio.enums import Resampling
import dask
import sys

In [3]:
temp_file_path = r"..\Data\Processed\Spatial_clustering\Temp"
dataset_dir = r"..\Data\Datasets"
sample_raster = rxr.open_rasterio(r"..\Data\Datasets\DEM_1km\DEM_1km_elevation_max.tif")

# Helper - clip and resample raster to ROI at 1km resolution
def clip_and_resample_raster(input_raster):
    resampled_raster = input_raster.rio.reproject_match(sample_raster, resampling= Resampling.bilinear)
    return resampled_raster

In [4]:
# Create precipitation climatology using ERA5L
era5L_files_path = r"C:\Users\A S U S\Documents\Study\ClimateData\CD-Workspace\Data\Datasets\ERA5L_prec"
era5L_files = sorted(glob.glob(os.path.join(era5L_files_path, "*.tif")))
era5L_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in era5L_files]

era5L_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True) for f in era5L_files], 
               dim=pd.Index(era5L_dates, name='time'), join = 'exact')

prec_climatology = era5L_files_rxr.resample(time='1YE').sum().mean(dim='time')
prec_clim = clip_and_resample_raster(prec_climatology)
prec_clim.rio.to_raster(os.path.join(temp_file_path, "prec_clim_resampled.tif"))

del era5L_files_rxr, prec_climatology, prec_clim

In [5]:
# Prepare fetch rasters
fetch_rasters_path = r"..\Data\Processed\Fetch"
fetch_rasters = ["fetch_dir_500m_resampled.tif", "fetch_dist_500m_resampled.tif", 
                 "fetch_dir_1000m_resampled.tif", "fetch_dist_1000m_resampled.tif"]

for fetch_raster in fetch_rasters:
    read_raster = rxr.open_rasterio(os.path.join(fetch_rasters_path, fetch_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(temp_file_path, fetch_raster))
    del read_raster, resampled_raster

In [6]:
# Create ERA5L secondary variables climatology
era5L_variables = ['dewpoint_temperature', 'soil_moisture', 'surface_pressure', 
                   'total_evaporation', 'wind_speed']

for variable in era5L_variables:
    var_path = os.path.join(dataset_dir, "ERA5L_2_" + variable)
    var_files = sorted(glob.glob(os.path.join(var_path, "*.tif")))
    var_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in var_files]

    var_files_rxr = xr.concat([rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}) for f in var_files], 
               dim=pd.Index(var_dates, name='time'), join = 'exact')
    var_climatology = var_files_rxr.resample(time='1YE').mean().mean(dim='time')
    var_clim = clip_and_resample_raster(var_climatology)
    var_clim.rio.to_raster(os.path.join(temp_file_path, "ERA5L_2_" + variable + "_clim_resampled.tif"))
    
    del var_files_rxr, var_climatology, var_clim

In [7]:
# Prepare DEM rasters
dem_rasters_path = r"..\Data\Datasets\DEM_1km"
dem_rasters = ['DEM_1km_aspect_mean.tif', 'DEM_1km_elevation_max.tif', 
               'DEM_1km_elevation_mean.tif', 'DEM_1km_elevation_min.tif',
               'DEM_1km_elevation_std.tif', 'DEM_1km_slope_mean.tif']

for dem_raster in dem_rasters:
    read_raster = rxr.open_rasterio(os.path.join(dem_rasters_path, dem_raster))
    resampled_raster = clip_and_resample_raster(read_raster)
    resampled_raster.rio.to_raster(os.path.join(temp_file_path, dem_raster))
    del read_raster, resampled_raster

In [4]:
# Prepare MODIS variables
from tqdm.notebook import tqdm 

modis_variables = ['lst_n', 'lst_d', 'evi', 'lst_diff']
for variable in tqdm(modis_variables, desc="Variables"):
    var_path = os.path.join(dataset_dir, "MODIS_2_" + variable)
    var_files = sorted(glob.glob(os.path.join(var_path, "*.tif")))
    var_dates = [pd.to_datetime(f[-12:-4], format='%Y%m%d') for f in var_files]
    
    df = pd.DataFrame({'file': var_files, 'date': var_dates})
    df['year'] = df['date'].dt.year

    yearly_means = []
    # Inner loop for years
    for year, group in tqdm(list(df.groupby('year')), desc=f"Years ({variable})", leave=False):
        # Resample each file first, then add time dimension
        resampled_arrays = []
        for f, date in zip(group['file'], group['date']):
            arr = rxr.open_rasterio(f, chunks={'x': 1024, 'y': 1024}, masked = True)
            arr_resampled = clip_and_resample_raster(arr)
            arr_resampled = arr_resampled.expand_dims(time=[date])  
            resampled_arrays.append(arr_resampled)
        
        ds = xr.concat(resampled_arrays, dim='time', join = 'exact')
        yearly_mean = ds.mean(dim='time', skipna=True, keep_attrs=True)
        yearly_means.append(yearly_mean)
        del resampled_arrays, ds

    stacked_means = xr.concat(yearly_means, dim='year')

    # Calculate climatological mean
    var_climatology = stacked_means.mean(dim='year', skipna=True)
    output_path = os.path.join(temp_file_path, "MODIS_2_" + variable + "_clim.tif")
    var_climatology.rio.to_raster(
        output_path,
        dtype='float32',
        compress='lzw',
        nodata = np.nan  
    )
    
    del df, yearly_means, var_climatology

Variables:   0%|          | 0/4 [00:00<?, ?it/s]

Years (lst_n):   0%|          | 0/20 [00:00<?, ?it/s]

Years (lst_d):   0%|          | 0/20 [00:00<?, ?it/s]

Years (evi):   0%|          | 0/20 [00:00<?, ?it/s]

Years (lst_diff):   0%|          | 0/20 [00:00<?, ?it/s]

In [ ]:
#print list of variables prepared
print("Prepared variables for spatial clustering:"
      "\n1. Precipitation Climatology (ERA5L)"
      "\n2. Fetch variables (dir and dist at 500m and 1000m)"
      "\n3. ERA5L secondary variables climatology (dewpoint_temperature, soil_moisture, surface_pressure, total_evaporation, wind_speed)"
      "\n4. DEM variables (aspect_mean, elevation_max, elevation_mean, elevation_min, elevation_std, slope_mean)"
      "\n5. MODIS variables climatology (evi, lst_diff, lst_n, lst_d)"
      "\n All variables saved to: Data\Processed\Spatial_clustering\Temp after resampling to 1km grid and reprojecting to WGS84")

Prepared variables for spatial clustering:
1. Precipitation Climatology (HAR)
2. Fetch variables (dir and dist at 500m and 1000m)
3. ERA5L secondary variables climatology (dewpoint_temperature, soil_moisture, surface_pressure, total_evaporation, wind_speed)
4. DEM variables (aspect_mean, elevation_max, elevation_mean, elevation_min, elevation_std, slope_mean)
5. MODIS variables climatology (evi, lst_diff, lst_n, lst_d)
 All variables saved to: Data\Processed\Spatial_clustering\Temp after resampling to 1km grid and reprojecting to WGS84
